# tropa-voice-mini — AI voice / voice-clone detector on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/tropa-voice-mini-ai-voice-detector.ipynb)

[`wasitaigeneratedcom/ai-voice-detector-small`](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small) is wasitaigenerated's open **tropa-voice-mini** detector (Apache-2.0, not gated). The card describes a fine-tuned [`facebook/wav2vec2-xls-r-300m`](https://huggingface.co/facebook/wav2vec2-xls-r-300m) backbone with a multi-layer attention head and the `audio-classification` pipeline. Weights are one **1.26 GB** F32 `model.safetensors`. Load it with the repo's own `detector.py` (downloaded via `hf_hub_download`, then `VoiceDetector.from_pretrained(repo)`). `det(path)` returns a calibrated `ai_score`, a `verdict` (`ai` / `uncertain` / `human`), and per-segment scores (**6 s** windows, **3 s** hop). Input is **16 kHz mono**; ffmpeg covers mp3/m4a/mp4.

**Why this model.** It is a small, CPU-friendly open detector for commercial TTS and voice clones, and it returns a timeline rather than one number. The [card](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small) reports **0.19% EER** on the authors' 14-system Commercial-Voices-2026 set, **13.5% EER** on Deepfake-Eval-2024, and **0.98% EER** on In-the-Wild (those figures are the card's own runs, not remeasured here). The hosted tropa-voice API is a different, larger system.

**Serving thresholds** live in `serving_head.json`. The card says the default `ai` threshold sits at a **0.5% false-positive rate** on held-out human speech (1% and 0.1% FPR cutoffs are listed too). Clips shorter than about **2 s** are less reliable.

**Licence:** Apache-2.0. Base model `facebook/wav2vec2-xls-r-300m` is also Apache-2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16** (`sm_75`), and **no FlashAttention 2**. This notebook keeps the card's **F32** weights (`detector.py` does not cast). The download is **1.26 GB**; peak VRAM for the short demos should be a few GB — an estimate from the shard plus 6 s windows, not a measured Colab run. The card also calls the model CPU-friendly (~0.06 s for a 30 s clip on GPU vs ~10 s on 4 CPU cores), so a CPU runtime still runs.

**Use cases:** [when to pick this notebook](https://github.com/donvito/notebooks/blob/main/docs/tropa-voice-mini-ai-voice-detector.md)

**Links:** [model card](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small) · [Commercial-Voices-2026-mini](https://huggingface.co/datasets/wasitaigeneratedcom/commercial-voices-2026-mini) · [benchmark code](https://github.com/wasitaigenerated/voice-deepfake-benchmark) · [announcement](https://x.com/wiagenerated/status/2119776813237096523)


## 1. Install

The [model card](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small) installs `torch`, `transformers`, `safetensors`, `soundfile`, and `huggingface_hub`, plus **ffmpeg** for mp3/m4a/mp4. Do **not** reinstall torch — keep Colab's CUDA wheel. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`.

`espeak-ng` is optional: a later cell synthesises one extra clip with it (same helper as the Granite Speech and Nemotron-3 Diarization notebooks). A missing binary skips that demo.


In [ ]:
%pip install -q "transformers>=4.40" safetensors soundfile huggingface_hub

# Card: pip install torch transformers safetensors soundfile huggingface_hub
# (+ ffmpeg for mp3/m4a/mp4). Do not replace Colab's CUDA torch wheel.
# Use %pip, not python -I -m pip — isolated mode drops Colab site-packages.
# Wav2Vec2Model is old enough that a 5.x pin is not required; 4.40+ is plenty.
# Do NOT install flash-attn (T4 cannot run FlashAttention 2).

import shutil
import subprocess

def _apt(*pkgs):
    missing = [p for p in pkgs if shutil.which(p) is None]
    if not missing:
        return
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", *missing], check=False)

_apt("ffmpeg")
if shutil.which("espeak-ng") is None and shutil.which("espeak") is None:
    _apt("espeak-ng")

print("ffmpeg:", shutil.which("ffmpeg") or "missing")
print("espeak:", shutil.which("espeak-ng") or shutil.which("espeak") or "missing (demo 3 will skip)")
print("install done — restart the runtime before the GPU-check cell if transformers was upgraded")


> **Restart the runtime now** if the install cell upgraded transformers: *Runtime > Restart session*, then continue from the next cell.
> Skip the restart if `transformers` was already ≥ 4.40.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. A T4 is the intended path. The card is CPU-friendly, so this cell continues on CPU instead of stopping.


In [ ]:
import gc
import os
import shutil
import subprocess
from pathlib import Path

import torch
import transformers

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
print("transformers", transformers.__version__, "| torch", torch.__version__)

if torch.cuda.is_available():
    DEVICE = "cuda"
    major, minor = torch.cuda.get_device_capability()
    free_b, total_b = torch.cuda.mem_get_info()
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:
        native_bf16 = major >= 8
    print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB")
    print("native bf16:", native_bf16, "| FlashAttention-2: off")
    print("dtype: float32 (detector.py keeps the F32 shard)")
    if major < 8:
        print("T4 path: F32 weights on sm_75. bfloat16 and FlashAttention-2 are unused.")
else:
    DEVICE = "cpu"
    print("device: cpu")
    print("No GPU — the card is CPU-friendly (~10 s for a 30 s clip on 4 cores).")
    print("A free T4 is faster (*Runtime > Change runtime type > T4 GPU*).")

REPO = "wasitaigeneratedcom/ai-voice-detector-small"
DATASET = "wasitaigeneratedcom/commercial-voices-2026-mini"
print("repo:", REPO)
print("ffmpeg:", shutil.which("ffmpeg") or "missing (soundfile fallback for FLAC/WAV)")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        free_now, total_now = torch.cuda.mem_get_info()
        print(
            f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
            f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.1f} GB"
            f"{suffix}"
        )
    else:
        import resource

        rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        print(f"CPU RSS peak: {rss_kb / 1024 / 1024:.2f} GB{suffix}")


## 3. Load tropa-voice-mini via `detector.py`

Same pattern as the [model card](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small): download `detector.py` with `hf_hub_download`, import it, then `VoiceDetector.from_pretrained(repo)`. That pulls `config.json`, `serving_head.json`, and the 1.26 GB `model.safetensors`. GPU if CUDA is up, else CPU. Public weights — cancel any Colab `HF_TOKEN` prompt; a token is optional for higher Hub rate limits.


In [ ]:
import importlib.util
import json

from huggingface_hub import hf_hub_download

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

detector_py = hf_hub_download(REPO, "detector.py")
spec = importlib.util.spec_from_file_location("tropa_voice_detector", detector_py)
detector_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(detector_mod)

det = detector_mod.VoiceDetector.from_pretrained(REPO, device=DEVICE)

n_params = sum(p.numel() for p in det.model.parameters())
head = det.head
print("parameters:", f"{n_params / 1e6:.1f}M  (card: XLS-R-300M backbone, 1.26 GB F32)")
print("device:", det.device, "| dtype:", next(det.model.parameters()).dtype)
print("window_seconds:", head.get("window_seconds"), "| hop_seconds:", head.get("hop_seconds"))
print("topk_fraction:", head.get("topk_fraction"), "| aggregation:", head.get("aggregation"))
print("verdict thresholds:", json.dumps(head.get("verdict"), indent=2))
print("FPR thresholds (card / serving_head.json):", json.dumps(head.get("thresholds"), indent=2))
print_vram("after load")


## 4. Demo — human vs commercial-TTS clips

A few English files from the public [`commercial-voices-2026-mini`](https://huggingface.co/datasets/wasitaigeneratedcom/commercial-voices-2026-mini) set the card links (2,084 synthetic clips from 14 commercial systems + 2,200 openly licensed real clips, 16 kHz mono FLAC). Dataset card: real clips from VoxPopuli (CC0) and LibriTTS / MLS / FLEURS (CC-BY-4.0); synthetic clips were generated through the providers' paid APIs. This is a smoke demo, not a reproduction of the card's EER table.

Each download is guarded: a missing file is skipped so the rest of the notebook still runs. Swap in your own 16 kHz clip later.


In [ ]:
import csv

try:
    from IPython.display import Audio, display
except ImportError:
    Audio = None

    def display(*_args, **_kwargs):
        pass

SAMPLE_DIR = Path("tropa_voice_samples")
SAMPLE_DIR.mkdir(exist_ok=True)

# Preferred English clips (duration > 6 s so a couple of windows exist).
# Human CC0 / CC-BY-4.0 rows and a few commercial-TTS systems named on the card.
PREFERRED = [
    {"file": "03856.flac", "want": "human"},  # CC0, ~14 s
    {"file": "02084.flac", "want": "human"},  # CC-BY-4.0, ~15 s
    {"file": "00135.flac", "want": "ai"},     # elevenlabs/tts/eleven-v4, ~13 s
    {"file": "01525.flac", "want": "ai"},     # openai/gpt-4o-mini-tts, ~15 s
    {"file": "00949.flac", "want": "ai"},     # minimax/speech-2.8-hd, ~23 s (timeline)
]


def try_hub_download(repo, filename, repo_type=None, dest=None):
    try:
        kwargs = {}
        if repo_type:
            kwargs["repo_type"] = repo_type
        path = hf_hub_download(repo, filename, **kwargs)
        if dest is not None:
            dest.parent.mkdir(parents=True, exist_ok=True)
            if Path(path).resolve() != dest.resolve():
                dest.write_bytes(Path(path).read_bytes())
            return dest
        return Path(path)
    except Exception as exc:
        print(f"skip {filename}: {type(exc).__name__}: {exc}")
        return None


def load_metadata():
    path = try_hub_download(DATASET, "metadata.csv", repo_type="dataset")
    if path is None:
        return []
    with path.open(newline="") as fh:
        return list(csv.DictReader(fh))


meta_rows = load_metadata()
by_file = {row["file"]: row for row in meta_rows}
print("metadata rows:", len(meta_rows) if meta_rows else "unavailable — using preferred filenames only")


def download_clip(filename):
    dest = SAMPLE_DIR / filename
    if dest.is_file() and dest.stat().st_size > 0:
        return dest
    path = try_hub_download(DATASET, f"audio/{filename}", repo_type="dataset", dest=dest)
    return path


loaded = []
for spec in PREFERRED:
    filename = spec["file"]
    path = download_clip(filename)
    if path is None:
        continue
    info = dict(by_file.get(filename) or {})
    info.update({"file": filename, "path": path, "want": spec["want"]})
    loaded.append(info)
    src = info.get("real_source_license") or info.get("system") or ""
    print(
        f"loaded {filename}  label={info.get('label', '?')}  "
        f"system={info.get('system', '?')}  {info.get('duration', '?')}s  {src}"
    )

if not loaded and meta_rows:
    print("preferred files missing — trying the next English human / AI rows from metadata")
    picks = []
    for want in ("human", "human", "ai", "ai"):
        for row in meta_rows:
            if row["label"] != want:
                continue
            if row.get("language") != "en":
                continue
            try:
                dur = float(row["duration"])
            except (TypeError, ValueError):
                continue
            if dur < 6:
                continue
            if row["file"] in {p["file"] for p in picks}:
                continue
            picks.append({**row, "want": want})
            break
    for spec in picks:
        path = download_clip(spec["file"])
        if path is None:
            continue
        spec = dict(spec)
        spec["path"] = path
        loaded.append(spec)
        print("fallback loaded", spec["file"], spec.get("label"), spec.get("system"))

print("clips ready:", len(loaded))


def bar(score, width=20):
    n = max(0, min(width, int(round(float(score) * width))))
    return "#" * n + "." * (width - n)


def show_result(result, title="", expected=None):
    if title:
        print(f"\n=== {title} ===")
    verdict = result["verdict"]
    score = result["ai_score"]
    extra = f"  (dataset label: {expected})" if expected else ""
    print(f"ai_score={score:.4f}  verdict={verdict}  duration={result.get('duration')}s{extra}")
    print(f"  {'':>8} {bar(score)}  {score:.4f}")
    segs = result.get("segments") or []
    print(f"  {len(segs)} segment(s)  (6 s window, 3 s hop)")
    for seg in segs:
        print(
            f"  {seg['start']:6.2f}–{seg['end']:6.2f}s  {bar(seg['ai_score'])}  "
            f"{seg['ai_score']:.4f}"
        )


results = []
for item in loaded:
    try:
        result = det(str(item["path"]))
    except Exception as exc:
        print(f"detect failed for {item['file']}: {type(exc).__name__}: {exc}")
        continue
    item["result"] = result
    results.append(item)
    title = f"{item['file']}  [{item.get('system') or item.get('want')}]"
    show_result(result, title=title, expected=item.get("label") or item.get("want"))
    try:
        display(Audio(filename=str(item["path"])))
    except Exception as exc:
        print("audio widget skipped:", exc)

if not results:
    print("No dataset clips scored. Demo 3 still tries a locally generated clip.")
print_vram("demo 1")


## 5. Demo — per-segment timeline

The file score is a calibrated top-k mean of window logits (`serving_head.json`: `aggregation=topk`, `topk_fraction=0.3`). Individual windows still show where the detector is more or less sure. This cell reuses the longest clip from the previous demo (the MiniMax ~23 s file if it loaded).


In [ ]:
timeline_item = None
if results:
    timeline_item = max(results, key=lambda item: float(item["result"].get("duration") or 0))

if timeline_item is None:
    print("No scored clip for a timeline. Skipping.")
else:
    result = timeline_item["result"]
    print(
        f"timeline clip: {timeline_item['file']}  "
        f"dataset={timeline_item.get('label')}  system={timeline_item.get('system')}  "
        f"{result.get('duration')}s"
    )
    show_result(result, title="file score", expected=timeline_item.get("label"))
    segs = result.get("segments") or []
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(8, 2.6))
        mids = [(s["start"] + s["end"]) / 2 for s in segs]
        scores = [s["ai_score"] for s in segs]
        widths = [max(0.05, s["end"] - s["start"]) for s in segs]
        colors = ["#d62728" if s >= head["verdict"]["ai"] else "#7f7f7f" if s > head["verdict"]["human"] else "#2ca02c" for s in scores]
        ax.bar(mids, scores, width=widths, color=colors, align="center", alpha=0.85)
        ax.axhline(head["verdict"]["ai"], color="#d62728", ls="--", lw=1, label=f"ai ≥ {head['verdict']['ai']:.3f} (0.5% FPR)")
        ax.axhline(head["verdict"]["human"], color="#2ca02c", ls="--", lw=1, label=f"human ≤ {head['verdict']['human']:.2f}")
        ax.set_ylim(0, 1.05)
        ax.set_xlabel("time (s)")
        ax.set_ylabel("segment ai_score")
        ax.set_title(timeline_item["file"])
        ax.legend(loc="lower right", fontsize=8)
        fig.tight_layout()
        plt.show()
    except Exception as exc:
        print("matplotlib skipped:", type(exc).__name__, exc)
        print("(text timeline is in the printout above)")

print_vram("demo 2")


## 6. Demo — score a clip generated here (optional)

Same `espeak-ng` helper as the [Granite Speech](granite-speech-5.0-2b-asr-translation.ipynb) and [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) notebooks: a short robotic line, long enough to clear the card's ~2 s warning. This is **not** a commercial voice clone; it only shows the detector on speech you just made. If `espeak-ng` is missing, the cell prints a skip and continues.


In [ ]:
espeak = shutil.which("espeak-ng") or shutil.which("espeak")
synth_path = SAMPLE_DIR / "espeak-demo.wav"

if espeak is None:
    print("espeak-ng not installed — skip demo 3. Dataset clips above are the main demo.")
else:
    text = (
        "This is a short computer generated voice for an open AI voice detection demo. "
        "It is not a commercial clone."
    )
    try:
        subprocess.run(
            [espeak, "-v", "en-us", "-s", "140", "-w", str(synth_path), text],
            check=True,
            capture_output=True,
        )
        print("wrote", synth_path, "with", espeak)
        synth = det(str(synth_path))
        show_result(synth, title="espeak-ng clip (generated in this notebook)")
        try:
            display(Audio(filename=str(synth_path)))
        except Exception as exc:
            print("audio widget skipped:", exc)
    except Exception as exc:
        print(f"espeak demo skipped: {type(exc).__name__}: {exc}")

print_vram("demo 3")


## 7. Caveats and cleanup

**What this notebook is for.** A free-T4 (or CPU) demo of tropa-voice-mini: the card's `detector.py` loader, calibrated `ai_score` / `verdict`, and the 6 s / 3 s hop timeline, on a handful of public Commercial-Voices-2026-mini files plus an optional `espeak-ng` clip.

**What it is not.** A reproduction of the card's EER table (those numbers are the authors' runs on full sets). A transcription or diarization model. A replacement for the hosted tropa-voice API, which the card reports as ahead on four of five listed benches.

**Card limitations (quoted, not retested).** Very short clips (<2 s), heavy background music, and phone-quality audio are harder for all detectors they compared. On MLAAD-tiny the card reports Spectra-AASIST ahead (0.71% vs 2.33% EER), from newer LLM-based TTS.

**API reminders.** `ai_score` is a calibrated probability in [0, 1]. Default `verdict=ai` uses the 0.5% FPR cutoff in `serving_head.json`. Review `detector.py` locally; it runs in this process. Input is 16 kHz mono.

**T4.** F32 weights, no FlashAttention-2, no extra CUDA kernels. `HF_TOKEN` is optional (public repo).

**Licence.** Apache-2.0. Demo FLACs follow the dataset card (CC0 / CC-BY-4.0 real speech; synthetic clips from paid APIs).

**Links again:** [model card](https://huggingface.co/wasitaigeneratedcom/ai-voice-detector-small) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/tropa-voice-mini-ai-voice-detector.md) · [dataset](https://huggingface.co/datasets/wasitaigeneratedcom/commercial-voices-2026-mini) · [announcement](https://x.com/wiagenerated/status/2119776813237096523)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
try:
    del det
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
else:
    print_vram("after cleanup")
    print("CPU run — no VRAM peak. On a T4, print peak VRAM from max_memory_allocated().")
print("tropa-voice-mini released. Re-run the load cell if you want another score.")
